## Introduction to Deep Learning: From Statistical Learning to Neural Networks

### Basics of Statistical Learning: A Foundation for Deep Learning and AI
Before diving into neural networks and deep learning, it’s essential to understand the roots of these techniques in statistical learning. At its core, machine learning is about learning patterns from data to make predictions or decisions without being explicitly programmed.

**Key Concepts**
- Supervised Learning: Learn a mapping from inputs `X` to outputs `Y` using labeled data.
- Regression: Predict continuous values (e.g., house prices).
- Classification: Predict discrete labels (e.g., spam vs. not spam).

### Learning Objectives

Upon successful completion of this session, you should be able to:
-   Fit a Simple Linear Regression model to data and interpret model coefficients
-   Build and train a neural network (one layer perceptron) to solve a regression problem, as an introduction to the concept of Artificial Neural Networks (ANNs)
-   Build and train a deep neural network using Keras -- a high-level, user-friendly API for building and training deep learning models.

### Simple Linear Regression: A Statistical Approach

#### Model definition

We start by defining a simple linear regression (SLR) statistical model:
The goal of a SLR model is to investigate the relationship between the **response** and the **predictor** variables.

Recall -- from high school algebra -- that the equation of a line describing a linear relation between $x$ and $y$ has the following algebraic form:

$y = b + mx$

where $m$ is the slope and $b$ is the y-intercept.

The general form of the SLR model -- for predicting a quantitative response (dependent) $Y$ on the basis of a single predictor (independent) variable $X$ -- closely resembles the equation of a line shown above, such that:

$Y = \beta_0 + \beta_1 X + \epsilon$

For an individual observation ($x_i, y_i$), the regression equation becomes:

$y_i = \beta_0 + \beta_1 x_i + \epsilon_i$

Where:
-   $\beta_0$ is the is the population y-intercept,
-   $\beta_1$ is the population slope,
-   $x_i$ is the *i*th (predictor/independent) observation, and
-   $\epsilon_i$ is the error or deviation of observation $y_i$ from the line $\beta_0 + \beta_1 x_i$,
-   and $\epsilon \sim N (0, \sigma^2)$

Together, $\beta_0$ and $\beta_1$ are known as the (unknown) population model ***coefficients*** or ***parameters***.

We use training data (or random sample) to produce estimates of the parameters -- $\hat\beta_0$ and $\hat\beta_1$ to describe the relation between $Y$ and $X$, **and make predictions of $\hat{y_i}$ given $x_i$**. Estimation is typically by the method of Ordinary Least Squares (OLS).

#### Errors (Loss)

The predicted (fitted) value of $Y$ ($\hat{y_i}$) based on the *i*th value of $X$ ($\hat{x_i}$) is obtained by:

$fit_i=\hat{y_i} = \hat\beta_0 + \hat\beta_1 x_i$

Then

$res_i=\epsilon_i = y_i - \hat{y_i}$

represents the *i*th residual -- this is the difference between the *i*th observed response value and the *i*th response value that is predicted by our linear model.

Loss is a measure of the difference between the actual values and our predictions. This difference is defined by Residual sum of squares (RSS) :

$RSS=\sum_{i=1}^{n}(y_i - \hat{y_i}){^2}$

With linear regression, it's common to use mean squared error (MSE), calculated using the formula:


$MSE  = \frac{1}{n}\sum_{i=1}^{n} \left(\hat{y}^{(i)} - y^{(i)}\right)^2$

Least square estimates of $\beta_0$ and $\beta_1$ are values of intercept and slope that minimize MSE.

**Why is this important?**

Linear regression introduces the idea of loss minimization (e.g., Mean Squared Error) and optimization, which are fundamental to neural networks. Neural networks extend regression by introducing non-linear transformations and multiple layers to learn complex patterns.



We will use `scikit‑learn`’s California Housing dataset, focusing on Median Income (`MedInc`) as the predictor and Median House Value (`MedHouseVal`) as the target.

**Why use `sklearn.datasets`?**
- Quick access to standard datasets for testing ML models.
- No need to manually download or preprocess common datasets.
- Compatible with scikit-learn pipelines.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score


import seaborn as sns
import statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.graphics.regressionplots import abline_plot


import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers


def load_california_medinc_target_from_fetch():
    """Fetch California Housing and return (MedInc, MedHouseVal) as arrays."""
    data = fetch_california_housing()
    X_full = data.data
    feature_names = data.feature_names  # ['MedInc','HouseAge','AveRooms',...]

    #get only median income feature, because we are investigating MedHouseVal ~ MedInc
    medinc_idx = feature_names.index('MedInc')
    X_medinc = X_full[:, medinc_idx].astype("float32").reshape(-1, 1)
    y = data.target.astype("float32").reshape(-1, 1)  # MedHouseVal, in 100k$
    return X_medinc, y


# ------------------------------------------------------------
# 1) Load data
# ------------------------------------------------------------

X_medinc, y = load_california_medinc_target_from_fetch()

# Wrap in a DataFrame for stats and correlation
df = pd.DataFrame({"MedInc": X_medinc.flatten(), "MedHouseVal": y.flatten()})

df

We fit a SLR model -- we estimate parameters that minimize errors

In [ ]:
# ------------------------------------------------------------
# 2) statsmodels simple linear regression
# ------------------------------------------------------------
#fit a SLR model to data
fit = smf.ols('MedHouseVal ~ MedInc', data=df).fit()
print(fit.summary())

Since the p-values associated with the estimated slope ($\beta{_1}$ of 0.4179) and the intercept ($\beta{_0}$ of 0.4509) are statistically significant, we can conclude that there is enough evidence to suggest that `MedInc` is a significant linear predictor of `MedHouseVal`.

Differently stated, **an increase of \\$10,000 (one unit) in median household income is associated associated with -- on average -- an increase of \\$41,790 (0.4179 X 100,000) in the median house value.**

In [ ]:
# Get the MSE value
mse_value = fit.mse_resid
print(f"The Mean Squared Error (mse_resid) is: {mse_value}")

In [ ]:
sns.scatterplot(data=df, x="MedInc", y="MedHouseVal")

## Regression with Perceptron
We will construct a neural network corresponding to a SLR model. We will train the network, implementing the gradient descent method.

### Model definition
We will start to use notation commonly used in machine learning, and define a SLR model as:

$\hat{y} = wx + b$

with the weigh $w$ corresponding to the slope $\beta_{1}$, and the bias $b$ corresponding to the intercept $\beta_{0}$


The simplest neural network model that describes the above relationship can be realized by using one **perceptron**. The **input** and **output** layers will have one **node** each ($x$ for input and $\hat{y} = z$ for output):

<img src="images/nn_model_linear_regression_simple.png" style="width:400px;">

**Weight** ($w$) and **bias** ($b$) are the parameters that will get updated when you **train** the model. They are initialized to some random values or set to 0 and updated as the training progresses.

**Weighted sum ($Z$)** is the weighted sum of the linear combinations of inputs, weights and bias, **before** the activation function is applied. We are not applying an activation function in this example, hence $Z$ is an estimate of $\hat{y}$.

For each training example ($x_i, y_i$), the prediction $\hat{y_i}$ can be calculated as:


$z_i =  w x_i + b,$

$\hat{y_i}  =  z_i,$


where $i = 1, \dots, m$.

We can organise all training examples as a vector $X$ of size ($1 \times m$) and perform scalar multiplication of $X$ ($1 \times m$) by a scalar $w$, adding $b$, which will be broadcasted to a vector of size ($1 \times m$):


$Z   = w X + b,$

$\hat{Y} = Z,$


This set of calculations is called **forward propagation**.

Given the estimate $\hat{Y}$, we can now measure the difference between the actual $y_i$ and the estimated $\hat{y_i}$ to obtain the errors (recall residuals) for each training example with the **loss function**:

$L\left(w, b\right)  = \frac{1}{2}\left(\hat{y_i} - y_i\right)^2$


To compare the resulting vector of the predictions $\hat{Y}$ ($1 \times m$) with the vector $Y$ of original values $y^{(i)}$, you can take an average of the loss function values for each of the training examples:

$\mathcal{L}\left(w, b\right)  = \frac{1}{2m}\sum_{i=1}^{m} \left(\hat{y_i} - y_i\right)^2$

This function is called the sum of squares **cost function**. The aim is to optimize the cost function during the training, which will minimize the differences between original values $y_i$ and predicted values $\hat{y_i}$.

When your weights were just initialized with some random values, and no training was done yet, you can't expect good results. You need to calculate the adjustments for the weight and bias, minimizing the cost function. This process is called **backward propagation**.

The method we will use to minimize the cost function is **gradient descent**, and we will attempt to identify parameter values that minimize the cost function by taking partial derivatives of the cost function.

<img src="images/gradient_descent.png" style="width:500px;">

We calculate partial derivatives as:


$\frac{\partial \mathcal{L} }{ \partial w } = \frac{1}{m}\sum_{i=1}^{m} \left(\hat{y_i} - y_i\right)x_i,$


$\frac{\partial \mathcal{L} }{ \partial b } = \frac{1}{m}\sum_{i=1}^{m} \left(\hat{y_i} - y_i\right)$

We then update the parameters iteratively using the expressions:


$w = w - \alpha \frac{\partial \mathcal{L} }{ \partial w },$

$b = b - \alpha \frac{\partial \mathcal{L} }{ \partial b },$

where $\alpha$ is the learning rate. Then repeat the process until the cost function stops decreasing.

The general **methodology** to build a neural network is to:
1. Define the neural network structure ( # of input units,  # of hidden units, etc).
2. Initialize the model's parameters
3. Loop:
    - Implement forward propagation (calculate the perceptron output),
    - Implement backward propagation (to get the required corrections for the parameters),
    - Update parameters.
4. Make predictions.


We first split the training set data into training and test sets, and standardize the data.

In [ ]:
# ------------------------------------------------------------
# 3) Train/test split; standardize MedInc using TRAIN stats
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_medinc, y, test_size=0.2, random_state=42
)
mean_medinc = X_train.mean()
std_medinc = X_train.std()
X_train_std = (X_train - mean_medinc) / std_medinc
X_test_std  = (X_test  - mean_medinc) / std_medinc

We then build the neuron

In [ ]:
# ------------------------------------------------------------
# 4) Keras one-neuron linear model (perceptron for regression)
# ------------------------------------------------------------
tf.keras.utils.set_random_seed(123)

model = keras.Sequential([
    layers.Dense(1, activation=None, input_shape=(1,))  # y_hat = w * x + b
])

#configure model for training
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.05),
              loss="mse",
              metrics=["mae"])

#train
history = model.fit(X_train_std, y_train, epochs=200, batch_size=256, verbose=1)

#evaluate the model's performance on the unseen test data
mse, mae = model.evaluate(X_test_std, y_test, verbose=0)

print(f"Keras (std MedInc) -> Test MSE={mse:.6f}, Test MAE={mae:.6f}")

model.summary()


In [ ]:
# Learned parameters in standardized space
w_std = float(model.layers[0].kernel.numpy()[0, 0])
b_std = float(model.layers[0].bias.numpy()[0])

# ------------------------------------------------------------
# 7) Map Keras params back to original MedInc units
# z = (MedInc - mean_medinc) / std_medinc
# y_hat = w_std * z + b_std
# => y_hat = (w_std/std_medinc) * MedInc + (b_std - w_std*mean_medinc/std_medinc)
# ------------------------------------------------------------
w_orig = w_std / float(std_medinc)
b_orig = b_std - w_std * float(mean_medinc) / float(std_medinc)
print(f"Keras parameters (original units) -> slope={w_orig:.6f}, intercept={b_orig:.6f}")

**Quiz**
- How do weights from the two approeaches (OLS vs Perceptron) compare?
- How do the MSEs from the two approaches compare?

## Let's Dive Deep! An Introduction to Deep Learning
Deep learning is a specific subfield of machine learning that involves learning representations from data with emphasis on learning successive layers of increasingly meaningful representations. The deep in deep learning stands for this idea of successive layers of representations. These layered representations are learned via models called neural networks, of whom perceptrons are building blocks.

Below is a representation of a two-layered dense neural network, with the aforementioned processes of forward and backward propagation shown.

<img src="images/dnn.png" style="width:720px;">

We want to build a model that can accurately predict the median house value in California districts, given a number of features from these districts.

**Features:** MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude

**Target:** MedHouseVal (median house value, in $100k units)


In [ ]:
"""
Dense fully connected regression with Keras on the California Housing dataset
using ALL features:
[MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude].

Outputs:
- Train/Val learning curves (loss & MAE)
- Final Test metrics: MSE, MAE, RMSE, R^2
"""

# ---------------------------------------------------------------------
# 0) Reproducibility
# ---------------------------------------------------------------------
tf.keras.utils.set_random_seed(123)
np.random.seed(123)

# ---------------------------------------------------------------------
# 1) Load data (downloads on first use)
# ---------------------------------------------------------------------
housing = fetch_california_housing(as_frame=True)
X = housing.data.astype("float32")
y = housing.target.astype("float32").values  # shape (n,)
feature_names = list(X.columns)
print("Features:", feature_names)
print("Target: MedHouseVal")

In [ ]:
# ---------------------------------------------------------------------
# 2) Train/Validation/Test splits
#    We'll do 70/15/15 by splitting twice.
# ---------------------------------------------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)  # 0.30 * 0.50 = 0.15

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

In [ ]:
# ---------------------------------------------------------------------
# 3) Feature scaling (fit on TRAIN only; transform val/test)
# ---------------------------------------------------------------------
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

**Building and training the model**

We define a dense model with three intermediate layers, each 64 units. The model ends with a single unit and no activation (it will be a linear layer).

The intermediate layers use a rectified linear unit (relu, zeroes out negative values) as their activation function.

These are the commonly used activations functions:

<img src="images/activation_functions.ppm.png" style="width:620px;">


In [ ]:
# ---------------------------------------------------------------------
# 4) Build Keras dense network
#    - 3 layers
#    - Hidden layers with ReLU
# ---------------------------------------------------------------------

model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(1) # Single output neuron for continuous output, no activation
])


#QUIZ!!
#compile model:
#Specify a lower learning rate, like 1e-3
#Use Adam Optimizer
#Use mean square error loss function
#uncomment and edit the code snippet below:

"""
model.compile(
        optimizer=
        loss=
        metrics=[keras.metrics.MAE, keras.metrics.RootMeanSquaredError(name="rmse")]
    )
"""

In [ ]:
# ---------------------------------------------------------------------
# 5) Train
# ---------------------------------------------------------------------
history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=200,
    batch_size=1024,
    verbose=0
)
model.summary()

In [ ]:
# ---------------------------------------------------------------------
# 6) Evaluate on test & compute R^2
# ---------------------------------------------------------------------
test_metrics = model.evaluate(X_test_s, y_test, verbose=0)
test_mse, test_mae, test_rmse = test_metrics
y_pred = model.predict(X_test_s, verbose=0).ravel()
test_r2 = r2_score(y_test, y_pred)

print(f"Test MSE : {test_mse:.4f}")
print(f"Test MAE : {test_mae:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test R^2 : {test_r2:.4f}")

In [ ]:
# ---------------------------------------------------------------------
# 7) Plot learning curves (loss & MAE)
# ---------------------------------------------------------------------
def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Loss
    axes[0].plot(history.history["loss"], label="train")
    axes[0].plot(history.history["val_loss"], label="val")
    axes[0].set_title("MSE Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)


    # MAE
    axes[1].plot(history.history["mean_absolute_error"], label="train")
    axes[1].plot(history.history["val_mean_absolute_error"], label="val")
    axes[1].set_title("MAE")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("MAE")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


plot_history(history)

**Quiz**
- Hyperparameter tuning: try different neural net widths/depths (e.g., [256,128,64]). How does this affect model's performance metrics?

### Tips & next steps

- Hyperparameter tuning: try different widths/depths (e.g., [256,128,64]), regularization strengths, and batch sizes.
- Target scaling: if you see slow convergence, you can standardize y as well and rescale outputs back for metrics.
- Cross‑validation: wrap training in K‑fold CV for more stable estimates.
- Production: export the scaler (StandardScaler) and Keras model together; at inference time, use the exact same scaler to transform incoming features before model.predict

We will wrap up with a complete, end‑to‑end Keras workflow that trains a dense fully connected network (multivariate regression) on the California Housing dataset using all eight features.

The script handles data loading, train/val/test splits, feature scaling, model definition with regularization and callbacks, training with early stopping, evaluation (MSE/MAE/RMSE/R²), and permutation importance to get a feel for which features matter most.

Below, we use two techniques to improve our model:
1. **Regularization** to improve generalization and reduce overfitting. This adds a penalty proportional to the sum of squared weights to the loss function, encrourages smaller weights, reducing model complexity and preventing overfitting.
2. **Dropout** -- randomly sets a fraction of layer outputs to zero during training (e.g., 20%). This prevents neurons from co-adapting too much, forcing the network to learn more robust representations.

In [ ]:
"""
Dense fully connected regression with Keras on the California Housing dataset
using ALL features:
[MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude].

Outputs:
- Train/Val learning curves (loss & MAE)
- Final Test metrics: MSE, MAE, RMSE, R^2
- Permutation importance (MAE delta per feature)
"""

# ---------------------------------------------------------------------
# 0) Reproducibility
# ---------------------------------------------------------------------
tf.keras.utils.set_random_seed(123)
np.random.seed(123)

# ---------------------------------------------------------------------
# 1) Load data (downloads on first use)
# ---------------------------------------------------------------------
housing = fetch_california_housing(as_frame=True)
X = housing.data.astype("float32")
y = housing.target.astype("float32").values  # shape (n,)
feature_names = list(X.columns)
print("Features:", feature_names)
print("Target: MedHouseVal")

# ---------------------------------------------------------------------
# 2) Train/Validation/Test splits
#    We'll do 70/15/15 by splitting twice.
# ---------------------------------------------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)  # 0.30 * 0.50 = 0.15

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# ---------------------------------------------------------------------
# 3) Feature scaling (fit on TRAIN only; transform val/test)
# ---------------------------------------------------------------------
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# ---------------------------------------------------------------------
# 4) Build Keras dense network
#    - Hidden layers with ReLU
#    - L2 weight regularization + Dropout
#    - BatchNormalization for stable training
# ---------------------------------------------------------------------
input_dim = X_train_s.shape[1]

def build_model():
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.20),

        layers.Dense(64, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.10),

        layers.Dense(32, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4)),

        layers.Dense(1, activation=None)  # Linear output for regression
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=[keras.metrics.MAE, keras.metrics.RootMeanSquaredError(name="rmse")]
    )
    return model

model = build_model()
model.summary()

# ---------------------------------------------------------------------
# 5) Callbacks: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
# ---------------------------------------------------------------------
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        "best_california_fc_model.keras",
        monitor="val_loss", save_best_only=True
    )
]

# ---------------------------------------------------------------------
# 6) Train
# ---------------------------------------------------------------------
history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=200,
    batch_size=1024,
    callbacks=callbacks,
    verbose=1
)

# ---------------------------------------------------------------------
# 7) Evaluate on test & compute R^2
# ---------------------------------------------------------------------
test_metrics = model.evaluate(X_test_s, y_test, verbose=0)
test_mse, test_mae, test_rmse = test_metrics
y_pred = model.predict(X_test_s, verbose=0).ravel()
test_r2 = r2_score(y_test, y_pred)

print(f"Test MSE : {test_mse:.4f}")
print(f"Test MAE : {test_mae:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test R^2 : {test_r2:.4f}")

# ---------------------------------------------------------------------
# 8) Plot learning curves (loss & MAE)
# ---------------------------------------------------------------------
def plot_history(history, out_path="california_fc_learning_curves.png"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Loss
    axes[0].plot(history.history["loss"], label="train")
    axes[0].plot(history.history["val_loss"], label="val")
    axes[0].set_title("MSE Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # MAE
    axes[1].plot(history.history["mean_absolute_error"], label="train")
    axes[1].plot(history.history["val_mean_absolute_error"], label="val")
    axes[1].set_title("MAE")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("MAE")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved learning curves -> {out_path}")

plot_history(history)

# ---------------------------------------------------------------------
# 9) Permutation importance (simple MAE delta per feature)
#    For each feature, shuffle its column in X_test_s and measure MAE change.
# ---------------------------------------------------------------------
def permutation_importance_mae(model, X_test_s, y_test, feature_names, n_repeats=3):
    base_pred = model.predict(X_test_s, verbose=0).ravel()
    base_mae = np.mean(np.abs(y_test - base_pred))
    results = []

    for j, fname in enumerate(feature_names):
        deltas = []
        for _ in range(n_repeats):
            X_perm = X_test_s.copy()
            # Shuffle ONLY column j
            rng = np.random.default_rng()
            X_perm[:, j] = rng.permutation(X_perm[:, j])
            perm_pred = model.predict(X_perm, verbose=0).ravel()
            perm_mae = np.mean(np.abs(y_test - perm_pred))
            deltas.append(perm_mae - base_mae)
        results.append((fname, float(np.mean(deltas)), float(np.std(deltas))))
    return sorted(results, key=lambda t: t[1], reverse=True)

perm_results = permutation_importance_mae(model, X_test_s, y_test, feature_names, n_repeats=5)
print("\nPermutation importance (MAE increase when shuffled):")
for fname, mean_delta, std_delta in perm_results:
    print(f"- {fname:10s}  ΔMAE = {mean_delta:.4f} ± {std_delta:.4f}")

# Save to CSV
pd.DataFrame(perm_results, columns=["feature", "mae_delta_mean", "mae_delta_std"]) \
  .to_csv("california_fc_permutation_importance.csv", index=False)
print("Saved permutation importance -> california_fc_permutation_importance.csv")

# ---------------------------------------------------------------------
# 10) (Optional) Save predictions vs. actuals for inspection
# ---------------------------------------------------------------------
pd.DataFrame({"y_test": y_test, "y_pred": y_pred}).to_csv("california_fc_predictions.csv", index=False)
print("Saved test predictions -> california_fc_predictions.csv")

# Done.

**Quiz**

- Change number of layers, and units per layer. How does this affect model accuracy?
- Train one model with regularization and one without, then compare validation loss and R²?